In [ ]:
import json, hashlib, shutil, sys, time
from pathlib import Path
from importlib.metadata import version
import kaggle_benchmarks as kbench
EXPECTED_SOURCE = "4b7deda714e222c1c0a1c10d0b0e31b85852a8828c855a8e9d3015b153d3c820"
PINS = {'pairs-uf-alpaca_eval': {'mapping.json': {'bytes': 39098, 'sha256': 'b82cb318eff25458e0b2850be0dd87c5f003a672a43868eafba87ef70b670121'}, 'pairs.jsonl': {'bytes': 569829, 'sha256': '318e86ee81746b1a8b9cab65dab22499098c9ede7694cccf2a5654e1776b6252'}}, 'pairs-hh-hh': {'mapping.json': {'bytes': 24399, 'sha256': 'e73badd65e1070cac674ff8cd126ce18d9763a3f8893aef55c8fdf6071afda9a'}, 'pairs.jsonl': {'bytes': 517387, 'sha256': '5231de0b8e042a347cff7dce09232e1d372413ffb7141d582cac0dcdc45ecdd1'}}, 'pairs-hh-alpaca_eval': {'mapping.json': {'bytes': 39098, 'sha256': 'aaa06951758e85bdeb1d4fd3ccb127b4b2e3e5e6cdf4dea4ef8ee87403bb3c62'}, 'pairs.jsonl': {'bytes': 478217, 'sha256': '3ccbd17930ade69cf880e3041f980c9cacc771cdb35053b660cbad56e94e6598'}}, 'pairs-uf-hh': {'mapping.json': {'bytes': 24399, 'sha256': 'e295fc41d60cda858fbb6ae7b7be60d432a9fb1929e7b9640829d6620edeb199'}, 'pairs.jsonl': {'bytes': 626724, 'sha256': '97214e1a590d34999ca04ef188194c98135743231fefea06fa0384e817c0dd61'}}}
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
manifests = [p for p in Path('/kaggle/input').rglob('source_manifest.json') if sha(p) == EXPECTED_SOURCE]
assert len(manifests) == 1, 'Expected exact reviewed source'
source = manifests[0].parent
for name, meta in json.loads(manifests[0].read_text())['files'].items():
    p = source/name
    assert p.stat().st_size == meta['bytes'] and sha(p) == meta['sha256'], name
# Reviewed dependency-only fix: keep immutable input source unchanged.
original_source = source
source = Path('/kaggle/working/judge_source')
shutil.copytree(original_source, source, dirs_exist_ok=True)
judging_path = source/'src/judging.py'
judging_text = judging_path.read_text()
assert judging_text.count('from src.training import _write_json') == 1
helper = r"""
def _write_json(path, value):
    '''Persist progress without importing the training environment.'''
    payload = json.dumps(value, sort_keys=True, indent=2, allow_nan=False) + '\n'
    with tempfile.NamedTemporaryFile(mode='w', dir=path.parent, encoding='utf-8', delete=False) as handle:
        temp = Path(handle.name)
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
    try:
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)
"""
judging_text = judging_text.replace('import time', 'import tempfile\nimport time', 1)
judging_text = judging_text.replace('from src.training import _write_json', helper.strip())
judging_path.write_text(judging_text)
assert sha(judging_path) == 'dd6938d008ac2c6de6e51c19925c2ca479329119c8f26298500680541604d36f'
sys.path.insert(0, str(source))
from src.judging import run_judging, parse_judgments
from src.pairwise import summarize
roots = [p.parent for p in Path('/kaggle/input').rglob('completed.json') if p.parent.name == 'matched_evaluation']
assert len(roots) == 1
matched = roots[0]
completion = json.loads((matched/'completed.json').read_text())
assert completion == {'status':'completed','responses':2100,'blind_pairs':1200,'decoding':{'do_sample':False,'max_new_tokens':256,'no_repeat_ngram_size':0,'repetition_penalty':1.1}}
for name, files in PINS.items():
    for filename, meta in files.items():
        p = matched/name/filename
        assert sha(p) == meta['sha256'] and p.stat().st_size == meta['bytes'], str(p)
    pairs = [json.loads(line) for line in (matched/name/'pairs.jsonl').read_text().splitlines()]
    assert len(pairs) == 300 and len({p['id'] for p in pairs}) == 300
    assert all(set(p) == {'id','instruction','response_a','response_b'} for p in pairs)
OUTPUT = Path('/kaggle/working/blind_judging')
OUTPUT.mkdir(exist_ok=True)
REQUESTED = 'google/gemini-3.6-flash'
setup = {'status':'verified_inputs','generation_version':357326672,'source_sha256':EXPECTED_SOURCE,'pairs':PINS,'requested_judge':REQUESTED,'sdk_version':version('kaggle-benchmarks'),'total_pairs':1200,'judging_dependency_fix_sha256':sha(source/'src/judging.py')}
setup_path = OUTPUT/'setup.json'
if setup_path.exists():
    assert json.loads(setup_path.read_text()) == setup, 'Resume identity mismatch'
else:
    setup_path.write_text(json.dumps(setup,indent=2)+'\n')
print('Verified immutable input: 1,200 blind pairs from version 357326672',flush=True)


In [ ]:
@kbench.task(name='pa2_saved_pair_judging')
def judge_saved_pairs(llm):
    # This task score measures completion only, never model quality.
    access = {'requested_judge':REQUESTED,'actual_judge':REQUESTED,'sdk_version':version('kaggle-benchmarks'),'status':'checking'}
    try:
        with kbench.chats.new('pa2_authored_access_check'):
            raw = llm.prompt('Compare answers to 2+2. A: 4. B: 5. Return only a JSON list: [{"id":"setup-1","decision":"A","reason":"brief explanation"}]. No markdown.')
        parsed = parse_judgments(raw, {'setup-1'})
        assert len(parsed) == 1 and parsed[0]['decision'] == 'A'
        access.update(status='verified',raw_response=raw)
    except Exception as exc:
        access.update(status='failed',error={'type':type(exc).__name__,'message':str(exc)})
        raise
    finally:
        (OUTPUT/'judge_access.json').write_text(json.dumps(access,indent=2)+'\n')
        print(json.dumps(access),flush=True)
    results = {}
    for name in ['pairs-hh-alpaca_eval','pairs-hh-hh','pairs-uf-alpaca_eval','pairs-uf-hh']:
        time.sleep(5)
        print('START '+name,flush=True)
        bank = 'hh' if name.endswith('-hh') else 'alpaca_eval'
        target = OUTPUT/name
        def request(prompt):
            with kbench.chats.new('pa2_anonymous_pairwise_attempt'):
                return llm.prompt(prompt)
        state = run_judging(matched/name/'pairs.jsonl',target,request,judge_model=REQUESTED,
                            batch_size=5,sleep_seconds=5,max_attempts=3,reuse=(target/'run.json').exists(),bank_kind=bank)
        def rows(p):
            return [json.loads(line) for line in p.read_text().splitlines()] if p.exists() else []
        results[name] = summarize(rows(matched/name/'pairs.jsonl'),json.loads((matched/name/'mapping.json').read_text()),
                                  rows(target/'judgments.jsonl'),rows(target/'failures.jsonl'),bootstrap_samples=2000,seed=42)
        (target/'comparison.json').write_text(json.dumps(results[name],indent=2)+'\n')
        (OUTPUT/'comparisons.json').write_text(json.dumps(results,indent=2)+'\n')
        print('FINISHED '+name+' '+state['status'],flush=True)
    states = {name:json.loads((OUTPUT/name/'run.json').read_text()) for name in results}
    total = sum(s['judged'] for s in states.values())
    final = {'status':'completed' if total==1200 else 'completed_with_failures','judged':total,'total':1200,'judge_model':REQUESTED,'generation_version':357326672,'score_is_completion_only':True}
    (OUTPUT/'completed.json').write_text(json.dumps(final,indent=2)+'\n')
    print(json.dumps(final),flush=True)
    return total/1200
judge_saved_pairs.run(llm=kbench.llms[REQUESTED])
